# 6D birleşik difüzyon (PDR tarzı) — geometri + renk TEK modelde

[*A Conditional Point Diffusion-Refinement Paradigm*, Lyu et al., ICLR 2022](https://arxiv.org/abs/2112.03530)
fikri: kısmi gözlem bir **koşul** olarak verilir, difüzyon tam bulutu üretir.

| | 2-aşamalı kol (`RePaint_part` / benchmark) | **bu notebook** |
|---|---|---|
| geometri | donmuş PoinTr üretir | **difüzyon üretir** |
| renk | ayrı DDPM, geometri sabitken | **aynı difüzyonda, birlikte** |
| koşullama | RePaint değiştirme (koşulsuz model) | kısmi buluta **koşullu** |
| aşama | 2 | 1 |

**Neden ilginç.** 2-aşamalıda renk, geometri dondurulduktan sonra geliyor — renk geometriyi
hiç etkileyemiyor. Oysa renk sınırı çoğu zaman parça sınırıdır, yani geometrik bir ipucudur.
6D birleşik difüzyon bu bilgi akışını açıyor.

**Kasıtlı tercih: PDR'ın resmi mimarisi yerine bizim EdgeConv omurgamız genişletildi.**
Onların repo'sunu kullansaydık "tek aşamalı mı iki aşamalı mı" sorusunun cevabı
"PointNet++ mü EdgeConv mü" sorusuna karışırdı. Aynı omurga, aynı şema, aynı metrik →
fark gerçekten paradigmadan gelir.

**PoinTr hiç kullanılmıyor** — geometri difüzyondan geliyor. Klon, CUDA extension ve 416 MB
checkpoint adımlarının hepsi bu notebook'ta yok.

---
### Yerelde CPU'da doğrulananlar (notebook'a girmeden önce)

* şekil testi — çıktı `(B,N,6)` ✓
* **overfit testi** — kayıp geo 1.009→0.128, rgb 1.032→0.012 ✓ model öğrenebiliyor
* **koşullama testi** — küre partial'ıyla örnekle → küreye mi yakın düzleme mi?
  400 epoch'ta başarısız, **2000 epoch'ta 2/2 başarılı**
* **modalite dengesizliği ölçüldü**: rgb hızlı yakınsıyor, geometri takılıyor.
  `LAM_GEO=4` küre Chamfer'ını 0.405 → 0.263'e indirdi. Varsayılan o yüzden 4.

---
# A · Kurulum

In [ ]:
import os, sys, subprocess, glob, json, time
for _r in ("/kaggle/temp", "/teamspace/studios/this_studio", "/content", os.getcwd()):
    if os.path.isdir(_r): WORK = _r; break
OUT_ROOT = "/kaggle/working/j6d_out" if os.path.isdir("/kaggle/working") else os.path.join(WORK, "j6d_out")
DATA_ROOT = os.path.join(WORK, "j6d_data")
for d in (OUT_ROOT, DATA_ROOT): os.makedirs(d, exist_ok=True)
os.environ.setdefault("HF_HOME", os.path.join(DATA_ROOT, "hf_cache"))
print("OUT_ROOT :", OUT_ROOT); print("DATA_ROOT:", DATA_ROOT)

subprocess.run("pip install -q open3d huggingface_hub wandb pandas", shell=True)
import numpy as np, torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
DEV = "cuda" if torch.cuda.is_available() else "cpu"
if DEV == "cpu": print("!! GPU YOK — cok yavas olur, Accelerator ayarini kontrol et")
subprocess.run("nvidia-smi --query-gpu=name --format=csv,noheader", shell=True)

---
# B · Weights & Biases

In [ ]:
import wandb
if not os.environ.get("WANDB_API_KEY"):
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
    except Exception: pass
_k = os.environ.get("WANDB_API_KEY")
wandb.login(key=_k) if _k else wandb.login()
WANDB_PROJECT, WANDB_ENTITY = "colored-pc-completion", None
print("wandb", wandb.__version__)

---
# C · Veri — 2-aşamalı kolla **aynı** kaynak

Aynı HF veri kümesi, aynı `occluded_occ` partial'ları, aynı train/test ayrımı.
İki kolun sonuçları ancak böyle karşılaştırılabilir.

In [ ]:
# ═════════════ AYARLAR ═════════════
HF_DATA      = "eylulpelinkilic/Colored_Point_Clouds"
CATEGORIES   = {"airplane": "02691156", "car": "02958343", "chair": "03001627"}
RUN_CATEGORIES = ["airplane"]
DIFFICULTIES = ["simple", "moderate", "hard"]
N_MODELS     = 200
N_PTS        = 2048          # uretilen tam bulut boyutu
N_PARTIAL    = 1024          # kosula verilen kismi bulut boyutu (FPS ile sabitlenir)
TEST_FRAC    = 0.25
EVAL_N       = 15
T_STEPS      = 200
EPOCHS       = 3000          # yerelde olculdu: kisa egitimde kosullama CALISMIYOR
LAM_GEO, LAM_RGB = 4.0, 1.0  # geometri agirligi: olculdu, chamfer 0.405->0.263
WIDTH, KNN, KNN_COND, NBLOCKS = 128, 16, 8, 3
CONDITIONAL  = True          # False -> kosulsuz + RePaint degistirme (ablasyon)
EXP_TAG      = "j6d"
FORCE_RETRAIN = False
# ═══════════════════════════════════

import numpy as np
from scipy.spatial import cKDTree
from huggingface_hub import snapshot_download

def srgb_to_lab(rgb):
    rgb=np.clip(rgb,0,1); lin=np.where(rgb>.04045,((rgb+.055)/1.055)**2.4,rgb/12.92)
    M=np.array([[.4124,.3576,.1805],[.2126,.7152,.0722],[.0193,.1192,.9505]])
    x=(lin@M.T)/np.array([.95047,1.,1.08883]); d=6/29
    f=np.where(x>d**3,np.cbrt(x),x/(3*d**2)+4/29)
    return np.stack([116*f[:,1]-16,500*(f[:,0]-f[:,1]),200*(f[:,1]-f[:,2])],1)
def deltaE(a,b): return np.linalg.norm(srgb_to_lab(a)-srgb_to_lab(b),axis=1)

def _fps(x, n, seed=0):
    """basit FPS — kismi bulutu sabit boyuta getirmek icin"""
    if len(x) == n:
        return x                                   # tam boyut: dokunma
    if len(x) < n:                                 # eksik: tekrarla doldurmak zorundayiz
        r = np.random.default_rng(seed)
        return np.concatenate([x, x[r.choice(len(x), n - len(x), replace=True)]])
    sel = np.zeros(n, np.int64); d = np.full(len(x), np.inf)
    sel[0] = np.random.default_rng(seed).integers(len(x))
    for i in range(1, n):
        d = np.minimum(d, ((x[:, :3] - x[sel[i-1], :3])**2).sum(1))
        sel[i] = d.argmax()
    return x[sel]

def load_split(category, difficulty):
    """-> [(gt6 (N_PTS,6) [-1,1], partial6 (N_PARTIAL,6) [-1,1], miss (N_PTS,), gt_rgb01, mid)]"""
    import open3d as o3d
    syn = CATEGORIES[category]
    root = snapshot_download(HF_DATA, repo_type="dataset",
        allow_patterns=[f"labeled_s3/{syn}/*", f"occluded_occ/{difficulty}/{syn}/*"])
    files = sorted(glob.glob(os.path.join(root, "labeled_s3", syn, "*.npz")))[:N_MODELS]
    out = []
    for j, f in enumerate(files):
        mid = os.path.splitext(os.path.basename(f))[0]
        pp = os.path.join(root, "occluded_occ", difficulty, syn, mid + ".ply")
        if not os.path.exists(pp): continue
        z = np.load(f); gxyz, grgb = z["xyz"].astype(np.float32), np.clip(z["rgb"],0,1).astype(np.float32)
        pxyz = np.asarray(o3d.io.read_point_cloud(pp).points, np.float32)
        d, idx = cKDTree(gxyz).query(pxyz, k=1)
        vis = np.zeros(len(gxyz), bool); vis[idx[d < 1e-6]] = True
        r = np.random.default_rng(j)
        s = r.choice(len(gxyz), N_PTS, replace=len(gxyz) < N_PTS)
        gxyz, grgb, miss = gxyz[s], grgb[s], (~vis)[s]
        if miss.sum() < 32 or (~miss).sum() < 32: continue
        c = gxyz.mean(0); sc = np.linalg.norm(gxyz - c, axis=1).max() + 1e-9
        gxyz = (gxyz - c) / sc
        gt6 = np.concatenate([gxyz, grgb*2-1], 1).astype(np.float32)     # [-1,1]
        par = _fps(gt6[~miss], N_PARTIAL, seed=j)
        out.append((gt6, par, miss, grgb, mid))
    return out
print("veri yukleyicisi hazir")

---
# D · Model — 6D koşullu denoiser

Yerelde CPU'da doğrulanmış kod, birebir. `PartialEncoder` kısmi bulutu **bir kez**
kodlar (difüzyon boyunca değişmez); denoiser her adımda üretilen bulutun kNN
grafiğini **yeniden kurar**, çünkü geometri artık hareket ediyor.

In [ ]:
import math
import numpy as np
import torch
import torch.nn as nn


# ══════════════════════════════════════════════════════════ difuzyon
def cosine_betas(T, s=0.008):
    t = torch.linspace(0, T, T + 1) / T
    f = torch.cos((t + s) / (1 + s) * math.pi / 2) ** 2
    ab = f / f[0]
    return (1 - ab[1:] / ab[:-1]).clamp(1e-8, 0.999)


class Diffusion:
    """Duz DDPM (eps-tahmini). Sinyal (N,6): [xyz | rgb], ikisi de [-1,1]'e olceklenmis."""

    def __init__(self, T=200, device="cpu"):
        self.T, self.device = T, device
        b = cosine_betas(T).to(device)
        a = 1.0 - b
        abar = torch.cumprod(a, 0)
        abar_prev = torch.cat([torch.ones(1, device=device), abar[:-1]])
        self.betas, self.alphas, self.abar = b, a, abar
        self.sqrt_abar, self.sqrt_1mabar = abar.sqrt(), (1 - abar).sqrt()
        self.post_var = b * (1 - abar_prev) / (1 - abar)
        self.post_c0 = b * abar_prev.sqrt() / (1 - abar)
        self.post_ct = (1 - abar_prev) * a.sqrt() / (1 - abar)

    def q_sample(self, x0, t, noise=None):
        noise = torch.randn_like(x0) if noise is None else noise
        sa = self.sqrt_abar[t].view(-1, *([1] * (x0.dim() - 1)))
        sb = self.sqrt_1mabar[t].view(-1, *([1] * (x0.dim() - 1)))
        return sa * x0 + sb * noise

    def p_sample(self, eps, x_t, t, generator=None):
        x0 = ((x_t - self.sqrt_1mabar[t] * eps) / self.sqrt_abar[t]).clamp(-1, 1)
        mean = self.post_c0[t] * x0 + self.post_ct[t] * x_t
        if t == 0:
            return mean
        z = torch.randn(x_t.shape, device=x_t.device, dtype=x_t.dtype, generator=generator)
        return mean + self.post_var[t].sqrt() * z


# ══════════════════════════════════════════════════════════ yardimcilar
def knn_idx(q, r, k, chunk=4096):
    """q (B,N,3) icindeki her nokta icin r (B,M,3) uzerindeki k komsu. q is r ise kendini de secer."""
    B, N, _ = q.shape
    M = r.shape[1]
    kk = min(k, M)
    out = torch.empty(B, N, kk, dtype=torch.long, device=q.device)
    for s in range(0, N, chunk):
        d = torch.cdist(q[:, s:s + chunk], r)
        out[:, s:s + chunk] = d.topk(kk, dim=-1, largest=False).indices
    if kk < k:                                   # cok kucuk bulut: tekrar ederek doldur
        out = out[..., [i % kk for i in range(k)]]
    return out


def gather_nb(feat, idx):
    """feat (B,M,C), idx (B,N,k) -> (B,N,k,C)."""
    B, M, C = feat.shape
    k = idx.shape[-1]
    off = (torch.arange(B, device=feat.device) * M).view(B, 1, 1)
    return feat.reshape(B * M, C)[(idx + off).reshape(-1)].reshape(B, idx.shape[1], k, C)


def timestep_embedding(t, dim):
    half = dim // 2
    f = torch.exp(-math.log(10000) * torch.arange(half, device=t.device).float() / half)
    a = t.float().view(-1, 1) * f.view(1, -1)
    return torch.cat([a.sin(), a.cos()], -1)


def _rel(q_xyz, r_xyz, idx):
    """Komsulara gore konum farki, YOGUNLUKTAN BAGIMSIZ olacak sekilde olceklenmis."""
    nb = gather_nb(r_xyz, idx)                       # (B,N,k,3)
    rel = nb - q_xyz.unsqueeze(2)
    d = rel.norm(dim=-1, keepdim=True)
    scale = d.mean(dim=(1, 2, 3), keepdim=True).clamp(min=1e-6)
    return torch.cat([rel / scale, d / scale], -1)   # (B,N,k,4)


# ══════════════════════════════════════════════════════════ kosul kodlayici
class PartialEncoder(nn.Module):
    """Kismi bulut P (B,M,6) -> nokta basina ozellik (B,M,C) + global (B,C).

    Kismi bulut difuzyon boyunca DEGISMEZ, o yuzden bir kez kodlanir ve
    butun adimlarda tekrar kullanilir (buyuk hizlanma)."""

    def __init__(self, width=128, k=16, n_blocks=2):
        super().__init__()
        self.k = k
        self.inp = nn.Linear(6, width)
        self.edges = nn.ModuleList([
            nn.Sequential(nn.Linear(2 * width + 4, width), nn.GELU(), nn.Linear(width, width))
            for _ in range(n_blocks)])

    def forward(self, P):
        idx = knn_idx(P[..., :3], P[..., :3], self.k)
        rel = _rel(P[..., :3], P[..., :3], idx)
        h = self.inp(P)
        for e in self.edges:
            hj = gather_nb(h, idx)
            hi = h.unsqueeze(2).expand_as(hj)
            h = h + e(torch.cat([hi, hj - hi, rel], -1)).max(2).values
        return h, h.max(1).values


# ══════════════════════════════════════════════════════════ denoiser
class Joint6DDenoiser(nn.Module):
    """eps-tahmini (B,N,6). Uretilen bulut + kismi buluta kosullu.

    conditional=False yaparsan kosullama kapanir -> KOSULSUZ 6D model olur
    (RePaint degistirmesiyle kullanilmak uzere). Ablasyon icin ayni sinif."""

    def __init__(self, width=128, k=16, k_cond=8, n_blocks=3, conditional=True):
        super().__init__()
        self.k, self.k_cond, self.width, self.conditional = k, k_cond, width, conditional
        self.enc = PartialEncoder(width, k) if conditional else None
        cin = 6 + (2 * width if conditional else 0)      # x_t + [yerel kosul | global kosul]
        self.inp = nn.Linear(cin, width)
        self.temb = nn.Sequential(nn.Linear(width, width), nn.SiLU(), nn.Linear(width, width))
        self.cond_mlp = (nn.Sequential(nn.Linear(width + 4, width), nn.GELU(),
                                       nn.Linear(width, width)) if conditional else None)
        self.edges = nn.ModuleList([
            nn.Sequential(nn.Linear(2 * width + 4, width), nn.GELU(), nn.Linear(width, width))
            for _ in range(n_blocks)])
        self.fuse = nn.ModuleList([
            nn.Sequential(nn.LayerNorm(2 * width), nn.Linear(2 * width, width), nn.GELU(),
                          nn.Linear(width, width)) for _ in range(n_blocks)])
        self.films = nn.ModuleList([nn.Linear(width, 2 * width) for _ in range(n_blocks)])
        self.out = nn.Sequential(nn.LayerNorm(width), nn.Linear(width, width), nn.GELU(),
                                 nn.Linear(width, 6))

    def encode_cond(self, P):
        """Kismi bulutu BIR KEZ kodla; ornekleme boyunca tekrar kullanilir."""
        if not self.conditional:
            return None
        f, g = self.enc(P)
        return dict(xyz=P[..., :3], feat=f, glob=g)

    def forward(self, x_t, t, cond=None):
        B, N, _ = x_t.shape
        xyz = x_t[..., :3]
        # geometri her adimda degisir -> grafik her cagrida yeniden kurulur
        idx = knn_idx(xyz, xyz, self.k)
        rel = _rel(xyz, xyz, idx)

        feats = [x_t]
        if self.conditional:
            assert cond is not None, "conditional=True ama cond verilmedi"
            ci = knn_idx(xyz, cond["xyz"], self.k_cond)         # uretilen -> kismi
            crel = _rel(xyz, cond["xyz"], ci)
            cnb = gather_nb(cond["feat"], ci)
            local = self.cond_mlp(torch.cat([cnb, crel], -1)).max(2).values
            feats += [local, cond["glob"].unsqueeze(1).expand(B, N, self.width)]
        h = self.inp(torch.cat(feats, -1))

        temb = self.temb(timestep_embedding(
            t.expand(B) if t.dim() else t.repeat(B), self.width))
        for e, fu, fi in zip(self.edges, self.fuse, self.films):
            hj = gather_nb(h, idx)
            hi = h.unsqueeze(2).expand_as(hj)
            loc = e(torch.cat([hi, hj - hi, rel], -1)).max(2).values
            glb = h.max(1, keepdim=True).values.expand_as(h)
            d = fu(torch.cat([loc, glb], -1))
            sc, sh = fi(temb).unsqueeze(1).chunk(2, -1)
            h = h + d * (1 + sc) + sh
        return self.out(h)


# ══════════════════════════════════════════════════════════ egitim
def train_joint(model, dif, pairs, epochs=300, bs=8, lr=2e-4, device="cpu",
                lam_geo=1.0, lam_rgb=1.0, log=50):
    """pairs: [(gt (N,6), partial (M,6))], ikisi de [-1,1]'e olceklenmis.

    Kayip modalite bazinda AYRI raporlanir: biri digerini bastiriyorsa gorunur olsun."""
    model.to(device).train()
    opt = torch.optim.AdamW(model.parameters(), lr, weight_decay=1e-4)
    n = len(pairs)
    hist = []
    for ep in range(epochs):
        perm = np.random.permutation(n)
        tot = np.zeros(2)
        for s in range(0, n, bs):
            b = [pairs[i] for i in perm[s:s + bs]]
            x0 = torch.stack([torch.as_tensor(g) for g, _ in b]).float().to(device)
            P = torch.stack([torch.as_tensor(p) for _, p in b]).float().to(device)
            t = torch.randint(0, dif.T, (len(b),), device=device)
            noise = torch.randn_like(x0)
            eps = model(dif.q_sample(x0, t, noise), t, model.encode_cond(P))
            l_geo = ((eps[..., :3] - noise[..., :3]) ** 2).mean()
            l_rgb = ((eps[..., 3:] - noise[..., 3:]) ** 2).mean()
            loss = lam_geo * l_geo + lam_rgb * l_rgb
            opt.zero_grad(); loss.backward(); opt.step()
            tot += np.array([l_geo.item(), l_rgb.item()]) * len(b)
        hist.append(tot / n)
        if log and (ep % log == 0 or ep == epochs - 1):
            print(f"    ep{ep:4d}  eps-MSE  geo {hist[-1][0]:.4f}  rgb {hist[-1][1]:.4f}", flush=True)
    return hist


# ══════════════════════════════════════════════════════════ ornekleme
@torch.no_grad()
def sample_joint(model, dif, partial, n_points, seed=0, device="cpu",
                 anchor_known=False, known=None, x_known=None):
    """Kismi buluta kosullu tam bulut uret -> (n_points, 6), [-1,1].

    anchor_known=True ise RePaint tarzi degistirme de uygulanir (gorunur noktalar
    her adimda geri enjekte edilir). Kosullu modelde bu OPSIYONEL; kosulsuz
    modelde (conditional=False) tamamlamanin tek yolu budur."""
    g = torch.Generator(device=device).manual_seed(seed)
    P = torch.as_tensor(partial).float().unsqueeze(0).to(device)
    model.eval().to(device)
    cond = model.encode_cond(P)
    x = torch.randn(1, n_points, 6, device=device, generator=g)
    m = xk = None
    if anchor_known:
        assert known is not None and x_known is not None
        m = torch.as_tensor(known).bool().view(1, -1, 1).to(device)
        xk = torch.as_tensor(x_known).float().unsqueeze(0).to(device) * m
    for t in range(dif.T - 1, -1, -1):
        eps = model(x, torch.tensor(t, device=device), cond)
        x = dif.p_sample(eps, x, t, generator=g)
        if anchor_known:
            noise = torch.randn(x.shape, device=device, generator=g)
            xkn = dif.q_sample(xk, torch.tensor([max(t - 1, 0)], device=device), noise)
            x = torch.where(m, xkn, x)
    return x[0].cpu().numpy()

---
# E · Eğitim + değerlendirme

In [ ]:
RESULTS = os.path.join(OUT_ROOT, f"results_{EXP_TAG}.jsonl")
def load_results():
    if not os.path.exists(RESULTS): return []
    o=[]
    for l in open(RESULTS):
        l=l.strip()
        if l:
            try: o.append(json.loads(l))
            except json.JSONDecodeError: pass
    return o
def append_result(r):
    with open(RESULTS,"a") as f: f.write(json.dumps(r)+"\n")
_done = {(r["category"],r["difficulty"],r["model"]) for r in load_results()}
print(f"onbellekte {len(_done)} sonuc | {RESULTS}")

def chamfer(a,b):
    d1,_=cKDTree(b).query(a,k=1); d2,_=cKDTree(a).query(b,k=1); return float(d1.mean()+d2.mean())

T0=time.time()
for category in RUN_CATEGORIES:
    print(f"\n{'='*60}\n  {category.upper()}  ({(time.time()-T0)/60:.0f} dk)\n{'='*60}", flush=True)
    DATA = load_split(category, DIFFICULTIES[0])
    n_test = max(1,int(len(DATA)*TEST_FRAC))
    TR, TE = list(range(len(DATA)-n_test)), list(range(len(DATA)-n_test, len(DATA)))
    print(f"  {len(DATA)} model | train {len(TR)} / test {len(TE)}", flush=True)

    DIF = Diffusion(T=T_STEPS, device=DEV)
    META = dict(cat=category, n_train=len(TR), n_pts=N_PTS, n_par=N_PARTIAL, T=T_STEPS,
                ep=EPOCHS, lam=[LAM_GEO,LAM_RGB], w=WIDTH, k=KNN, kc=KNN_COND,
                nb=NBLOCKS, cond=CONDITIONAL)
    ck = os.path.join(OUT_ROOT, f"{category}_{EXP_TAG}.pt")
    model = Joint6DDenoiser(width=WIDTH,k=KNN,k_cond=KNN_COND,n_blocks=NBLOCKS,
                            conditional=CONDITIONAL)
    loaded=False
    if os.path.exists(ck) and not FORCE_RETRAIN:
        try:
            z=torch.load(ck,map_location=DEV,weights_only=False)
            if z.get("meta")==META: model.load_state_dict(z["sd"]); model.to(DEV); loaded=True
        except Exception: pass
    if loaded: print("  model checkpointten yuklendi", flush=True)
    else:
        print(f"  -- egitim ({EPOCHS} epoch, lam_geo={LAM_GEO})", flush=True)
        pairs=[(DATA[i][0], DATA[i][1]) for i in TR]
        h=train_joint(model, DIF, pairs, epochs=EPOCHS, bs=8, lr=2e-4, device=DEV,
                      lam_geo=LAM_GEO, lam_rgb=LAM_RGB, log=max(EPOCHS//10,1))
        torch.save({"sd":model.state_dict(),"meta":META}, ck)

    for diff in DIFFICULTIES:
        D = load_split(category, diff)
        idxs=[i for i in (TE if EVAL_N is None else TE[:EVAL_N]) if i < len(D)]
        todo=[i for i in idxs if (category,diff,D[i][4]) not in _done]
        print(f"\n  -- {diff}: {len(idxs)} test, {len(todo)} yapilacak", flush=True)
        run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY,
            id=f"j6d-{category}-{diff}", name=f"joint6d/{category}/{diff}",
            group="joint6d", job_type="eval", resume="allow", reinit=True,
            config=dict(arm="joint6d", category=category, difficulty=diff,
                        conditional=CONDITIONAL, epochs=EPOCHS, T=T_STEPS,
                        lam_geo=LAM_GEO, lam_rgb=LAM_RGB, width=WIDTH,
                        n_pts=N_PTS, n_partial=N_PARTIAL, dataset=HF_DATA))
        t1=time.time()
        for n,i in enumerate(todo):
            gt6, par, miss, grgb, mid = D[i]
            gen = sample_joint(model, DIF, par, N_PTS, seed=1000+n, device=DEV,
                               anchor_known=not CONDITIONAL,
                               known=None if CONDITIONAL else np.zeros(N_PTS,bool),
                               x_known=None if CONDITIONAL else gt6)
            gen_xyz, gen_rgb = gen[:,:3], np.clip((gen[:,3:]+1)/2, 0, 1)
            gt_xyz = gt6[:,:3]
            # --- geometri ---
            ch_gen = chamfer(gen_xyz, gt_xyz)
            ch_par = chamfer(par[:,:3], gt_xyz)          # referans: hic tamamlama yok
            # --- renk: EKSIK bolgedeki GT noktasinin en yakin URETILEN noktasi ---
            _, gi = cKDTree(gen_xyz).query(gt_xyz[miss], k=1)
            de_j6d = float(deltaE(gen_rgb[gi], grgb[miss]).mean())
            # --- ayni geometri uzerinde NN-kopya baseline'i ---
            vis_xyz, vis_rgb = gt_xyz[~miss], grgb[~miss]
            _, vj = cKDTree(vis_xyz).query(gen_xyz, k=1)
            de_nn = float(deltaE(vis_rgb[vj][gi], grgb[miss]).mean())
            rec=dict(category=category, difficulty=diff, model=mid,
                     chamfer_gen=ch_gen, chamfer_partial=ch_par,
                     dE_joint6d=de_j6d, dE_nn_on_gen=de_nn)
            append_result(rec); _done.add((category,diff,mid))
            run.log({f"model/{k}":v for k,v in rec.items() if isinstance(v,float)})
            print(f"    [{n+1}/{len(todo)}] {mid[:10]} chamfer {ch_gen:.4f} (partial {ch_par:.4f}) "
                  f"| dE j6d {de_j6d:5.2f} | NN {de_nn:5.2f}  ({(time.time()-t1)/(n+1):.0f}s)", flush=True)
        rs=[r for r in load_results() if r["category"]==category and r["difficulty"]==diff]
        if rs:
            agg={k:float(np.mean([r[k] for r in rs])) for k in
                 ("chamfer_gen","chamfer_partial","dE_joint6d","dE_nn_on_gen")}
            agg["chamfer_vs_partial_x"]=agg["chamfer_partial"]/max(agg["chamfer_gen"],1e-9)
            agg["n"]=len(rs)
            run.log(agg); run.summary.update(agg)
            print(f"    => chamfer {agg['chamfer_gen']:.4f} vs partial {agg['chamfer_partial']:.4f} "
                  f"({agg['chamfer_vs_partial_x']:.2f}x) | dE j6d {agg['dE_joint6d']:.2f} "
                  f"| NN {agg['dE_nn_on_gen']:.2f}  (n={len(rs)})", flush=True)
        # gorsel
        gt6,par,miss,grgb,_ = D[idxs[0]]
        gen = sample_joint(model, DIF, par, N_PTS, seed=0, device=DEV)
        c3=lambda x,c: wandb.Object3D(np.concatenate([x, np.clip(c,0,1)*255],1).astype(np.float32))
        run.log({"cloud/gt": c3(gt6[:,:3], grgb),
                 "cloud/partial": c3(par[:,:3], (par[:,3:]+1)/2),
                 "cloud/generated": c3(gen[:,:3], (gen[:,3:]+1)/2)})
        run.finish()
print(f"\nBITTI — {(time.time()-T0)/60:.0f} dk")

---
# F · Özet + rapor

In [ ]:
import pandas as pd
rs=load_results(); assert rs, "sonuc yok"
df=pd.DataFrame(rs)
g=(df.groupby(["category","difficulty"])
     .agg(n=("model","count"), chamfer_gen=("chamfer_gen","mean"),
          chamfer_partial=("chamfer_partial","mean"),
          dE_joint6d=("dE_joint6d","mean"), dE_nn_on_gen=("dE_nn_on_gen","mean")).reset_index())
g["chamfer_vs_partial_x"]=g.chamfer_partial/g.chamfer_gen
g=g.sort_values(["category","difficulty"],key=lambda s:s.map({"simple":0,"moderate":1,"hard":2}).fillna(s))
pd.set_option("display.width",200,"display.max_columns",30)
print(g.to_string(index=False,float_format=lambda x:f"{x:.3f}"))
g.to_csv(os.path.join(OUT_ROOT,"joint6d_grid.csv"),index=False)

s=wandb.init(project=WANDB_PROJECT,entity=WANDB_ENTITY,name="SUMMARY-joint6d",
             id="SUMMARY-joint6d",resume="allow",job_type="summary",reinit=True)
s.log({"grid":wandb.Table(dataframe=g)}); s.finish()
print("\n->", os.path.join(OUT_ROOT,"joint6d_grid.csv"))

---
## Nasıl okunmalı

**1. `chamfer_vs_partial_x` > 1 mi?** Üretilen geometri ham partial'dan iyi mi. 1'in altındaysa
model geometriyi öğrenememiş demektir — bu durumda renk sayıları da anlamsızdır, önce
`EPOCHS`'u artır.

**2. `dE_joint6d` vs `dE_nn_on_gen`.** İkisi de **aynı üretilen geometri** üzerinde ölçülüyor,
yani fark tamamen renklendirmeden geliyor. Difüzyonun rengi, naif kopyalamadan iyi mi?

**3. 2-aşamalı kolla karşılaştırma.** `Multi_dataset_benchmark`'ın `chamfer/completion_to_gt`
(PoinTr) değeriyle buradaki `chamfer_gen`'i karşılaştır. Beklenti: PoinTr daha iyi, çünkü
ShapeNet-55'te on binlerce şekille eğitildi, biz kategori başına ~150 modelle geometri
üretmeyi öğretiyoruz. **Bu beklenen bir sonuç ve dürüstçe raporlanmalı** — "tek aşamalı
yaklaşım daha az veriyle geometriyi öğrenemiyor" da bir bulgudur.

**4. Ablasyon.** `CONDITIONAL = False` yapıp tekrar çalıştır: koşullama yerine RePaint
değiştirmesi kullanılır. Aradaki fark tam olarak **"koşullama mı değiştirme mi"** sorusunu
cevaplar — bu RePaint literatürüne doğrudan bir katkı.

Yerel testte ölçülen ve buraya taşınan iki ayar: `EPOCHS` kısa olduğunda koşullama
**hiç çalışmıyor** (400'de 0/2, 2000'de 2/2), ve `LAM_GEO=4` geometriyi belirgin
iyileştiriyor. İkisini de düşürme.